# Notebook 02: Huis Data Exploratie 

In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path.cwd().parent / "locatie_data"

In [3]:
df_huis = pd.read_parquet(
    DATA_DIR / "huis788.parquet"
)

### Algemene controle

In [4]:
print("Shape:", df_huis.shape)
print("Chronologisch:", df_huis["timestamp"].is_monotonic_increasing)

df_huis.info()

Shape: (3057890, 26)
Chronologisch: True
<class 'pandas.DataFrame'>
RangeIndex: 3057890 entries, 0 to 3057889
Data columns (total 26 columns):
 #   Column        Dtype                           
---  ------        -----                           
 0   timestamp     datetime64[us, Europe/Amsterdam]
 1   CO2           float64                         
 2   CTYPE         str                             
 3   ENAME         str                             
 4   ENE           float64                         
 5   EONNAME       str                             
 6   ETYPE         str                             
 7   GNAME         str                             
 8   POW           float64                         
 9   source_zip    str                             
 10  source_csv    str                             
 11  CO2_IMP       float64                         
 12  COST_IMP      float64                         
 13  CTYPEC        str                             
 14  ENE_CNT_IMP   float6

In [5]:
df_huis.head()

,timestamp,CO2,CTYPE,ENAME,ENE,EONNAME,ETYPE,GNAME,POW,source_zip,...,CO2_EXP,COST,COST_EXP,ENE_CNT_EXP,ENE_EXP,CO2EQ,ESTYPE,forecast.ENE,forecast.POW,PRICE
0,2024-03-08 01:00:00+01:00,NaN,electricity,boiler-16eda211b788,NaN,eiot-16eda211b788,boiler,SERENE,6.504839,240308_boiler-16eda211b788_7813337caa64460dd37...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-03-08 01:00:00+01:00,NaN,electricity,hoofdmeter-16eda211b788,NaN,eiot-16eda211b788,powermeter,SERENE,97.580645,240308_hoofdmeter-16eda211b788_583371188e6bd6b...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-03-08 01:00:00+01:00,NaN,electricity,overig-16eda211b788,NaN,eiot-16eda211b788,powermeter,SERENE,34.007143,240308_overig-16eda211b788_c9a2174e7ac825737c3...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-03-08 01:00:00+01:00,NaN,electricity,totaalverbruik-16eda211b788,NaN,eiot-16eda211b788,powermeter,SERENE,100.000000,240308_totaalverbruik-16eda211b788_15b395d52b6...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-03-08 01:00:00+01:00,NaN,electricity,wcdskeukeneilandenvaatwasser-16eda211b788,NaN,eiot-16eda211b788,dishwasher,SERENE,42.476190,240308_wcdskeukeneilandenvaatwasser-16eda211b7...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Overzicht van de meetreeksen

In [6]:
# maakt per ENAME een samenvatting vd timestamps 
overzicht = (df_huis.groupby("ENAME").agg(
        aantal_rijen=("timestamp", "size"), # telt hoeveel rijen er per ENAME zijn 
        unieke_timestamps=("timestamp", "nunique"), # telt hoeveel verschillende timestamps er per ENAME zijn 
        eerste_meting=("timestamp", "min"), # eerste timestamp die voorkomt per ENAME 
        laatste_meting=("timestamp", "max") # laatste timestamp die voorkomt per ENAME
    )
)

# berekent hoeveel extra rijen er zijn doordat timestamps vaker dan 1 keer voorkomen 
overzicht["dubbele_timestamps"] = (overzicht["aantal_rijen"] - overzicht["unieke_timestamps"])

overzicht.sort_values("aantal_rijen", ascending=False)

,aantal_rijen,unieke_timestamps,eerste_meting,laatste_meting,dubbele_timestamps
ENAME,,,,,
overig-16eda211b788,398790,230149,2024-03-08 01:00:00+01:00,2026-09-10 01:55:00+02:00,168641
totaalverbruik-16eda211b788,397154,228499,2024-03-08 01:00:00+01:00,2026-09-10 01:55:00+02:00,168655
wasmachine-16eda211b788,381248,214270,2024-03-08 01:05:00+01:00,2026-09-10 01:55:00+02:00,166978
omvormer-16eda211b788,368720,200703,2024-03-08 06:55:00+01:00,2026-09-10 01:55:00+02:00,168017
boiler-16eda211b788,362465,195413,2024-03-08 01:00:00+01:00,2026-09-10 01:55:00+02:00,167052
wcdskeukeneilandenvaatwasser-16eda211b788,361006,194555,2024-03-08 01:00:00+01:00,2026-09-10 01:55:00+02:00,166451
hoofdmeter-16eda211b788,318076,188249,2024-03-08 01:00:00+01:00,2026-09-07 08:05:00+02:00,129827
priceele_c911efafaa,171217,171217,2025-01-14 09:20:00+01:00,2026-09-10 01:55:00+02:00,0
co2ele_c911efafaa,167079,167070,2024-06-20 12:15:00+02:00,2026-09-10 01:55:00+02:00,9


### Totaalverbruik bekijken

In [7]:
TOTAALVERBRUIK = "totaalverbruik-16eda211b788"

# aparte dataframe met alleen de rijen van totaalverbruik
df_totaal = (df_huis[df_huis["ENAME"] == TOTAALVERBRUIK].copy()
             .sort_values("timestamp").reset_index(drop=True)
)

print("Aantal rijen:", len(df_totaal))
print("Unieke timestamps:", df_totaal["timestamp"].nunique())

# berekent hoeveel extra rijen er zijn doordat timestamps vaker voorkomen
print("Dubbele timestamps:",
    len(df_totaal) - df_totaal["timestamp"].nunique()
)

df_totaal.head()

Aantal rijen: 397154
Unieke timestamps: 228499
Dubbele timestamps: 168655


,timestamp,CO2,CTYPE,ENAME,ENE,EONNAME,ETYPE,GNAME,POW,source_zip,...,CO2_EXP,COST,COST_EXP,ENE_CNT_EXP,ENE_EXP,CO2EQ,ESTYPE,forecast.ENE,forecast.POW,PRICE
0,2024-03-08 01:00:00+01:00,NaN,electricity,totaalverbruik-16eda211b788,NaN,eiot-16eda211b788,powermeter,SERENE,100.0,240308_totaalverbruik-16eda211b788_15b395d52b6...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-03-08 01:05:00+01:00,1.533333,electricity,totaalverbruik-16eda211b788,8.333333,eiot-16eda211b788,powermeter,SERENE,40.0,240308_totaalverbruik-16eda211b788_15b395d52b6...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-03-08 01:10:00+01:00,0.613333,electricity,totaalverbruik-16eda211b788,3.333333,eiot-16eda211b788,powermeter,SERENE,45.0,240308_totaalverbruik-16eda211b788_15b395d52b6...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-03-08 01:15:00+01:00,0.690000,electricity,totaalverbruik-16eda211b788,3.750000,eiot-16eda211b788,powermeter,SERENE,NaN,240308_totaalverbruik-16eda211b788_15b395d52b6...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-03-08 01:20:00+01:00,NaN,electricity,totaalverbruik-16eda211b788,NaN,eiot-16eda211b788,powermeter,SERENE,45.0,240308_totaalverbruik-16eda211b788_15b395d52b6...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df_totaal["timestamp"].value_counts().value_counts().sort_index()

count
1     59844
2    168655
Name: count, dtype: int64

In [9]:
print(df_huis.columns.tolist())

['timestamp', 'CO2', 'CTYPE', 'ENAME', 'ENE', 'EONNAME', 'ETYPE', 'GNAME', 'POW', 'source_zip', 'source_csv', 'CO2_IMP', 'COST_IMP', 'CTYPEC', 'ENE_CNT_IMP', 'ENE_IMP', 'CO2_EXP', 'COST', 'COST_EXP', 'ENE_CNT_EXP', 'ENE_EXP', 'CO2EQ', 'ESTYPE', 'forecast.ENE', 'forecast.POW', 'PRICE']


In [10]:
print(df_totaal.columns.tolist())
df_totaal.head()

['timestamp', 'CO2', 'CTYPE', 'ENAME', 'ENE', 'EONNAME', 'ETYPE', 'GNAME', 'POW', 'source_zip', 'source_csv', 'CO2_IMP', 'COST_IMP', 'CTYPEC', 'ENE_CNT_IMP', 'ENE_IMP', 'CO2_EXP', 'COST', 'COST_EXP', 'ENE_CNT_EXP', 'ENE_EXP', 'CO2EQ', 'ESTYPE', 'forecast.ENE', 'forecast.POW', 'PRICE']


,timestamp,CO2,CTYPE,ENAME,ENE,EONNAME,ETYPE,GNAME,POW,source_zip,...,CO2_EXP,COST,COST_EXP,ENE_CNT_EXP,ENE_EXP,CO2EQ,ESTYPE,forecast.ENE,forecast.POW,PRICE
0,2024-03-08 01:00:00+01:00,NaN,electricity,totaalverbruik-16eda211b788,NaN,eiot-16eda211b788,powermeter,SERENE,100.0,240308_totaalverbruik-16eda211b788_15b395d52b6...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-03-08 01:05:00+01:00,1.533333,electricity,totaalverbruik-16eda211b788,8.333333,eiot-16eda211b788,powermeter,SERENE,40.0,240308_totaalverbruik-16eda211b788_15b395d52b6...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-03-08 01:10:00+01:00,0.613333,electricity,totaalverbruik-16eda211b788,3.333333,eiot-16eda211b788,powermeter,SERENE,45.0,240308_totaalverbruik-16eda211b788_15b395d52b6...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-03-08 01:15:00+01:00,0.690000,electricity,totaalverbruik-16eda211b788,3.750000,eiot-16eda211b788,powermeter,SERENE,NaN,240308_totaalverbruik-16eda211b788_15b395d52b6...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-03-08 01:20:00+01:00,NaN,electricity,totaalverbruik-16eda211b788,NaN,eiot-16eda211b788,powermeter,SERENE,45.0,240308_totaalverbruik-16eda211b788_15b395d52b6...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
